# Football Performance AI – Exploration

This notebook demonstrates how to use the football performance prediction system.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import load_all_data
from src.models.ensemble import EnsembleModel
from src.scenarios.engine import WhatIfEngine

In [ ]:
# Load data
data = load_all_data('../data/sample')
print('Players:', len(data['players']))
print('Teams:', len(data['teams']))
print('Leagues:', len(data['leagues']))
print('Stat rows:', len(data['player_stats']))

In [ ]:
# Rating distribution by position
merged = data['player_stats'].merge(data['players'][['id', 'position']], left_on='player_id', right_on='id')
merged.groupby('position')['rating'].describe().round(2)

In [ ]:
# Load trained ensemble
ensemble = EnsembleModel()
ensemble.load('../models_saved')
engine = WhatIfEngine(ensemble, data)
print('Models loaded.')

In [ ]:
# Trajectory example
player_id = data['players'].iloc[0]['id']
traj = engine.predict_age_trajectory(int(player_id), age_range=(18, 38))

ages = [t.age for t in traj]
ratings = [t.predicted_rating for t in traj]
ci_lo = [t.confidence_low for t in traj]
ci_hi = [t.confidence_high for t in traj]

plt.figure(figsize=(10, 5))
plt.plot(ages, ratings, 'b-o', label='Predicted Rating')
plt.fill_between(ages, ci_lo, ci_hi, alpha=0.2, label='95% CI')
plt.xlabel('Age')
plt.ylabel('Rating')
plt.title(f'Age Trajectory – Player {player_id}')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()